In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import time
import math
import matplotlib.pyplot as plt
import datetime

In [2]:
tkel_bs = 11
puppi_bs = 14
nlargestpt_per_puppipart = 5
npuppipart = 5
puppicands = nlargestpt_per_puppipart * npuppipart

In [6]:
train_path = 'train_data.pt'
test_path = 'test_data.pt'

In [7]:
train_data = torch.load(train_path, weights_only=True)
test_data = torch.load(test_path, weights_only=True)

x_train, y_train = train_data['x'], train_data['y'].float()
x_test, y_test = test_data['x'], test_data['y'].float()


In [10]:
# Count the number of tkel features in data
data_features = train_data['features']
tkel_index = sum([feature.startswith('tkel') for feature in data_features])
if tkel_index != tkel_bs:
    raise(f'Required exact match of tkel features: {tkel_bs}, found {tkel_index}')
puppi_features = sum([feature.startswith('mpuppi') for feature in data_features])
if puppi_features // puppicands != puppi_bs:
    raise(f'Required exact match of puppi features: {puppi_bs}, '\
            'found {puppi_features}')

# Split into tkel and puppi
tkel_train = x_train[:, 0:tkel_bs]
puppi_train_flat = x_train[:, tkel_bs:]
if puppi_features%puppicands == 0:
    puppi_train = puppi_train_flat.reshape(-1, puppicands, puppi_bs)
else:
    raise UnboundLocalError(f"Unresolved splitting of puppi cands, found "\
            f"features {puppi_features} for {puppicands} puppi canddiates")

tkel_test = x_test[:, 0:tkel_bs]
puppi_test_flat = x_test[:, tkel_bs:]
if puppi_features%puppicands == 0:
    puppi_test = puppi_test_flat.reshape(-1, puppicands, puppi_bs)
else:
    raise UnboundLocalError(f"Unresolved splitting of puppi cands, found "\
            f"features {puppi_features} for {puppicands} puppi canddiates")



In [30]:
# Extract unnormalized pT (which is the 0-th feature of the 14 puppi features)
puppi_pt_train = puppi_train[:, :, 0].clone()
puppi_pt_test = puppi_test[:, :, 0].clone()

In [31]:
# Compute Normalization statistics on Train only
tkel_mean = tkel_train.mean(dim=0)
tkel_std = tkel_train.std(dim=0)
tkel_std[tkel_std < 1e-6] = 1.0 # Prevent division by zero

# Normalize all 25 candidates uniformly
puppi_train_reshaped = puppi_train.reshape(-1, 14)
puppi_mean = puppi_train_reshaped.mean(dim=0)
puppi_std = puppi_train_reshaped.std(dim=0)
puppi_std[puppi_std < 1e-6] = 1.0

In [40]:
# Apply normalization
tkel_train_norm = (tkel_train - tkel_mean) / tkel_std
tkel_test_norm = (tkel_test - tkel_mean) / tkel_std

puppi_train_norm = (puppi_train - puppi_mean) / puppi_std
puppi_test_norm = (puppi_test - puppi_mean) / puppi_std



In [ ]:
tkel_expand = tkel_train_norm.unsqueeze(1).expand(-1, puppicands, -1)

tensor([[[ 0.2552, -1.7602, -1.3970,  ...,  0.9916,  0.3859, -0.5058],
         [ 0.2552, -1.7602, -1.3970,  ...,  0.9916,  0.3859, -0.5058],
         [ 0.2552, -1.7602, -1.3970,  ...,  0.9916,  0.3859, -0.5058],
         ...,
         [ 0.2552, -1.7602, -1.3970,  ...,  0.9916,  0.3859, -0.5058],
         [ 0.2552, -1.7602, -1.3970,  ...,  0.9916,  0.3859, -0.5058],
         [ 0.2552, -1.7602, -1.3970,  ...,  0.9916,  0.3859, -0.5058]],

        [[ 0.0980,  0.7159,  0.1876,  ..., -1.0084,  0.3859,  0.1405],
         [ 0.0980,  0.7159,  0.1876,  ..., -1.0084,  0.3859,  0.1405],
         [ 0.0980,  0.7159,  0.1876,  ..., -1.0084,  0.3859,  0.1405],
         ...,
         [ 0.0980,  0.7159,  0.1876,  ..., -1.0084,  0.3859,  0.1405],
         [ 0.0980,  0.7159,  0.1876,  ..., -1.0084,  0.3859,  0.1405],
         [ 0.0980,  0.7159,  0.1876,  ..., -1.0084,  0.3859,  0.1405]],

        [[-0.7507, -1.3767,  1.3419,  ..., -1.0084, -1.9531, -0.0240],
         [-0.7507, -1.3767,  1.3419,  ..., -1

In [46]:
tkel_train_norm.shape

torch.Size([197754, 11])